# 03 — KPI Validation
This notebook shows how to validate SQL-style KPIs using Pandas.

**Concepts demonstrated:** merge, derived columns, groupby aggregation, KPI validation

In [ ]:
import pandas as pd
from pathlib import Path

## 1. Load source data

In [ ]:
RAW = Path('../data/raw')
orders = pd.read_csv(RAW / 'orders.csv', parse_dates=['order_date'])
order_items = pd.read_csv(RAW / 'order_items.csv')

orders.head()

## 2. Recreate order-level revenue

In [ ]:
oi = order_items.copy()
oi['revenue'] = oi['quantity'] * oi['item_price']
order_revenue = oi.groupby('order_id', as_index=False).agg(
    order_revenue=('revenue', 'sum'),
    total_items=('quantity', 'sum')
)
order_revenue.head()

## 3. Merge with orders and calculate monthly KPIs

In [ ]:
orv = orders.merge(order_revenue, on='order_id', how='left')
recognized = orv[orv['order_status'].isin(['paid', 'shipped', 'delivered'])].copy()
recognized['order_month'] = recognized['order_date'].dt.to_period('M').astype(str)

monthly_kpis = recognized.groupby('order_month').agg(
    orders=('order_id', 'nunique'),
    revenue=('order_revenue', 'sum')
).reset_index().sort_values('order_month')
monthly_kpis['aov'] = monthly_kpis['revenue'] / monthly_kpis['orders']
monthly_kpis

## 4. Validate top products by revenue

In [ ]:
product_revenue = oi.groupby('product_id', as_index=False)['revenue'].sum().sort_values('revenue', ascending=False)
product_revenue.head(10)

## 5. Export outputs for BI tools (optional)

In [ ]:
OUT = Path('../data/processed')
OUT.mkdir(parents=True, exist_ok=True)
monthly_kpis.to_csv(OUT / 'monthly_kpis_validated.csv', index=False)
product_revenue.to_csv(OUT / 'product_revenue_validated.csv', index=False)
print('Saved validation files to', OUT)

## Try it yourself

### Questions
1. Compare monthly KPI output with your SQL query results.
2. Validate top 5 customers by spend.
3. Add refunded revenue as a separate monthly metric.

### Answers

In [ ]:
# 1. Monthly KPI output is already in monthly_kpis above. Compare it with your SQL export using merge if needed.
# Example if you have SQL output saved as CSV:
# sql_kpis = pd.read_csv('../data/processed/dw_monthly_revenue_metrics.csv')
# compare = monthly_kpis.merge(sql_kpis, left_on='order_month', right_on='order_month', how='inner')
# display(compare.head())

# 2. Top 5 customers by spend
top_customers = recognized.groupby('customer_id', as_index=False)['order_revenue'].sum().sort_values('order_revenue', ascending=False).head(5)
display(top_customers)

# 3. Refunded revenue as a separate monthly metric
refunded = orv[orv['order_status'] == 'refunded'].copy()
refunded['order_month'] = refunded['order_date'].dt.to_period('M').astype(str)
monthly_refunded = refunded.groupby('order_month', as_index=False)['order_revenue'].sum().rename(columns={'order_revenue': 'refunded_revenue'})
display(monthly_refunded.head())